# ENAD — Adversarial Image Detector (Gradio demo)

Upload a 32x32 image; the app extracts the same layer features used in training, scores the
selected detectors, and the logistic head predicts **adversarial** vs **normal**.

- **ENAD** = the LID + Mahalanobis + OCSVM baseline.
- **ENAD-GA** = the GA-selected subset for that (dataset, attack).

Single-image **Mahalanobis is exact** (input-preprocessing + best magnitude, as in training);
single-image **LID is approximate** (a single image has no in-batch reference, so the train set
stands in). Everything else (OCSVM + supervised detectors) is scored exactly as in training.

**Requirements:** GPU T4, Internet ON. **Add data:** `enad-demo-assets` (NB6),
`enad-ocsvm-pkl`, `enad-pkl`, and the weights dataset.

In [1]:
import os
import sys
import json
import pickle
import logging
import subprocess

import numpy as np
import torch

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import PredefinedSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (precision_recall_curve, roc_auc_score,
                             accuracy_score, auc)

UPSTREAM = "deep_Mahalanobis_detector"
UPSTREAM_URL = "https://github.com/pokaxpoka/deep_Mahalanobis_detector.git"
DATA_ROOT = "/kaggle/working/data"
TRAIN_CAP = None     
                    
CV_FOLDS  = 5      

models = None         
data_loader = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def setup(seed: int = 0, clone: bool = True):
    global models, data_loader, DEVICE
    if clone and not os.path.exists(UPSTREAM):
        subprocess.run(["git", "clone", "--quiet", UPSTREAM_URL], check=True)
    sys.path.append("./" + UPSTREAM)
    from deep_Mahalanobis_detector import models as _m, data_loader as _dl
    models, data_loader = _m, _dl
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return models, data_loader


CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

def n_classes(ds_name):  return 100 if ds_name == "cifar100" else 10
def n_layers(net_type):  return 5 if net_type == "resnet" else 4


def _logger(name):
    lg = logging.getLogger(name)
    if not lg.handlers:
        lg.setLevel(logging.INFO)
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s: %(message)s", "%H:%M:%S"))
        lg.addHandler(h)
        lg.propagate = False         
    return lg


def aupr(y_true, y_pred, pos_label=1):
    precision, recall, _ = precision_recall_curve(y_true, y_pred, pos_label=pos_label)
    return auc(recall, precision)

import torchvision.transforms as _T

def get_model_transforms(net_type, ds_name, num_classes,
                         weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth"):
    ckpt = os.path.join(weights_dir, f"{net_type}_{ds_name}.pth")
    if net_type == "resnet":
        model = models.ResNet34(num_c=num_classes)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        tf = _T.Compose([_T.ToTensor(), _T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    else:
        raise ValueError(f"unsupported net_type: {net_type}")
    model.to(DEVICE).eval()
    return model, tf


def extract_activations(X, model, layer_idx, return_pred=False):
    with torch.no_grad():
        acts = model.intermediate_forward(X, layer_idx)
    acts = acts.view(acts.size(0), acts.size(1), -1).mean(2)
    if return_pred:
        with torch.no_grad():
            y_pred = model(X).argmax(1)
        return acts, y_pred
    return acts


def activations_from_loader(model, layer, loader):
    acts, labels = [], []
    for x, y in loader:
        acts.append(extract_activations(x.to(DEVICE), model, layer).cpu().numpy())
        labels.append(y.cpu().numpy())
    return np.concatenate(acts), np.concatenate(labels)

class Datasets:
    def __init__(self, ds_name, in_transform, net_type, adv_type, outf, batch_size=100):
        self.ds_name, self.net_type, self.adv_type = ds_name, net_type, adv_type
        self.train_loader, _ = data_loader.getTargetDataSet(
            ds_name, batch_size, in_transform, DATA_ROOT)
        if TRAIN_CAP and len(self.train_loader.dataset) > TRAIN_CAP:
            from torch.utils.data import DataLoader as _DL, Subset as _Subset
            _idx = np.random.RandomState(0).choice(
                len(self.train_loader.dataset), TRAIN_CAP, replace=False)
            self.train_loader = _DL(_Subset(self.train_loader.dataset, _idx),
                                    batch_size=batch_size, shuffle=False)

        tag = f"{net_type}_{ds_name}_{adv_type}"
        clean = torch.load(f"{outf}/clean_data_{tag}.pth", map_location="cpu")
        new_size = batch_size * (len(clean) // batch_size)
        clean = clean[:new_size]
        noisy = torch.load(f"{outf}/noisy_data_{tag}.pth", map_location="cpu")[:new_size]
        adv = torch.load(f"{outf}/adv_data_{tag}.pth", map_location="cpu")[:new_size]
        targets = torch.load(f"{outf}/label_{tag}.pth", map_location="cpu").numpy()[:new_size]

        self.X_test = torch.cat([adv, clean, noisy]).to(DEVICE)
        self.y_test = np.tile(targets, 3)
        self.adv_test = np.array([0] * len(adv) + [1] * (len(clean) + len(noisy)))

        p_size = len(clean)
        p_split = int(p_size * 0.1)
        idxs_trainval = np.concatenate([
            np.arange(p_split),
            np.arange(p_size, p_size + p_split),
            np.arange(2 * p_size, 2 * p_size + p_split)])
        self.idxs_test = np.delete(np.arange(len(self.X_test)), idxs_trainval)
        pivot = int(len(idxs_trainval) / 6)
        self.idxs_train = np.concatenate([idxs_trainval[:pivot],
                                          idxs_trainval[2*pivot:3*pivot],
                                          idxs_trainval[4*pivot:5*pivot]])
        self.idxs_val = np.concatenate([idxs_trainval[pivot:2*pivot],
                                        idxs_trainval[3*pivot:4*pivot],
                                        idxs_trainval[5*pivot:]])


class DatasetsGA(Datasets):
    def __init__(self, ds_name, in_transform, net_type, adv_type, outf, batch_size=100):
        super().__init__(ds_name, in_transform, net_type, adv_type, outf, batch_size)
        split_size = len(self.X_test) // 3
        (self.idxs_train, self.idxs_val,
         self.idxs_ga_val, self.idxs_test) = self._split(split_size)

    @staticmethod
    def _split(split_size, trainval_frac=0.1, ga_frac=0.1):
        tv = int(split_size * trainval_frac)
        ga = int(split_size * ga_frac)
        idxs_trainval = np.concatenate([
            np.arange(tv),
            np.arange(split_size, split_size + tv),
            np.arange(2*split_size, 2*split_size + tv)])
        idxs_ga_val = np.concatenate([
            np.arange(tv, tv + ga),
            np.arange(split_size + tv, split_size + tv + ga),
            np.arange(2*split_size + tv, 2*split_size + tv + ga)])
        used = np.concatenate([idxs_trainval, idxs_ga_val])
        idxs_test = np.delete(np.arange(split_size * 3), used)
        pivot = int(len(idxs_trainval) / 6)
        idxs_train = np.concatenate([idxs_trainval[:pivot],
                                     idxs_trainval[2*pivot:3*pivot],
                                     idxs_trainval[4*pivot:5*pivot]])
        idxs_val = np.concatenate([idxs_trainval[pivot:2*pivot],
                                   idxs_trainval[3*pivot:4*pivot],
                                   idxs_trainval[5*pivot:]])
        return idxs_train, idxs_val, idxs_ga_val, idxs_test
        
class _ActLoader:
    def __init__(self, model, ds, idxs_attr, batch_size=100):
        self.model, self.ds, self.idxs_attr, self.bs = model, ds, idxs_attr, batch_size
    def __call__(self, layer_idx):
        X = self.ds.X_test[getattr(self.ds, self.idxs_attr)]
        acts, y = [], []
        for batch in torch.split(X, self.bs):
            a, yp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            acts.append(a.cpu().numpy()); y.append(yp.cpu().numpy())
        return np.concatenate(acts), np.concatenate(y)

def LabelledTrainLoader(model, ds, batch_size=100): return _ActLoader(model, ds, "idxs_train", batch_size)
def LabelledValLoader(model, ds, batch_size=100):   return _ActLoader(model, ds, "idxs_val", batch_size)
def LabelledGAValLoader(model, ds, batch_size=100): return _ActLoader(model, ds, "idxs_ga_val", batch_size)
def LabelledTestLoader(model, ds, batch_size=100):  return _ActLoader(model, ds, "idxs_test", batch_size)

class TrainValLoader:
    def __init__(self, model, ds, batch_size=100):
        self.model, self.ds, self.bs = model, ds, batch_size
    def __call__(self, layer_idx):
        X_train, y_train = activations_from_loader(self.model, layer_idx, self.ds.train_loader)
        adv_train = np.repeat(1, len(X_train))
        X_valid = self.ds.X_test[self.ds.idxs_val]
        av, yv = [], []
        for batch in torch.split(X_valid, self.bs):
            a, yp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            av.append(a.cpu().numpy()); yv.append(yp.cpu().numpy())
        X_valid = np.concatenate(av); y_valid = np.concatenate(yv)
        adv_valid = self.ds.adv_test[self.ds.idxs_val]
        return X_train, X_valid, y_train, y_valid, adv_train, adv_valid

class GroupedScaler(BaseEstimator, TransformerMixin):
    def __init__(self, with_centering=True):
        self.with_centering = with_centering
    def fit(self, X, y=None):
        groups = X[:, -1].astype(int); X = X[:, :-1]
        if self.with_centering:
            self.group_means = [np.mean(X[(groups == lab).nonzero()[0]], axis=0)
                                for lab in np.unique(groups)]
        return self
    def transform(self, X, y=None):
        groups = X[:, -1].astype(int); X = X[:, :-1]
        if not self.with_centering:
            return X
        old_idxs, X_norm = [], []
        for lab in np.unique(groups):
            m = (groups == lab).nonzero()[0]
            X_norm.extend(X[m] - self.group_means[lab]); old_idxs.extend(m)
        return np.array(X_norm)[np.argsort(old_idxs)]
    def get_params(self, deep=True): return {"with_centering": self.with_centering}
    def set_params(self, **p):
        for k, v in p.items(): setattr(self, k, v)
        return self
        
class _BaseTrainer:
    def __init__(self, n_layers, detector_class, exp_name="", outf="",
                 logger=None, pre_computed=False, pre_computed_path=None):
        self.n_layers = n_layers
        self.detector_class = detector_class
        self.exp_name = exp_name
        self.outf = outf
        self.logger = logger or _logger(exp_name or "enad")
        self.pre_computed = pre_computed
        self.pre_computed_path = pre_computed_path

    def fit(self, dl_train, dl_unseen_train, adv_unseen_train):
        self.detectors = self._train_layer_detectors(dl_train)
        train_scores = self.get_layer_scores(dl_unseen_train)
        self.lr = self._train_logistic(train_scores, adv_unseen_train)
        return self

    def predict(self, data_loader):
        scores = self.get_layer_scores(data_loader)
        preds = self.lr.predict(scores)
        probas = self.lr.predict_proba(scores)
        return scores, np.array([preds, probas[:, 0]]).T

    def _train_logistic(self, X, adv):
        lr = LogisticRegressionCV(penalty="l1", solver="liblinear",
                                  max_iter=10000, n_jobs=-1)
        lr.fit(X, adv)
        return lr

class NetDetector(_BaseTrainer):
    def _train_layer_detectors(self, data_loader):
        detectors = []
        for li in range(self.n_layers):
            X_train, X_valid, y_train, y_valid, adv_train, adv_valid = data_loader(li)
            X = np.concatenate((X_train, X_valid)); y = np.concatenate((y_train, y_valid))
            adv = np.concatenate((adv_train, adv_valid))
            if self.pre_computed:
                with open(self.pre_computed_path) as f:
                    params = json.load(f)
                det = clone(self.detector_class).set_params(
                    **params[f"{self.exp_name}_{li}"]).fit(np.c_[X_train, y_train])
            else:
                from skopt.callbacks import DeltaYStopper
                srch = clone(self.detector_class)
                srch.fit(np.c_[X, y], adv, callback=DeltaYStopper(0.02, n_best=10))
                with open(f"{self.outf}/bayes_{self.exp_name}_{li}.pkl", "wb") as f:
                    pickle.dump(srch, f, pickle.HIGHEST_PROTOCOL)
                det = srch.estimator.set_params(**srch.best_params_).fit(np.c_[X_train, y_train])
                self.logger.info(f"{self.exp_name} L{li}: {srch.best_params_}")
            detectors.append(det)
        return detectors

    def get_layer_scores(self, data_loader):
        scores = []
        for li in range(self.n_layers):
            X, y = data_loader(li)
            scores.append(self.detectors[li].decision_function(np.c_[X, y]))
        return np.vstack(scores).T


class SupervisedDetector(_BaseTrainer):
    def _train_layer_detectors(self, data_loader):
        detectors = []
        for li in range(self.n_layers):
            X_train, X_valid, y_train, y_valid, adv_train, adv_valid = data_loader(li)
            X = np.concatenate((X_train, X_valid)); y = np.concatenate((y_train, y_valid))
            adv = np.concatenate((adv_train, adv_valid))
            if self.pre_computed and self.pre_computed_path:
                with open(self.pre_computed_path) as f:
                    params = json.load(f)
                det = clone(self.detector_class).set_params(
                    **params.get(f"{self.exp_name}_{li}", {})).fit(np.c_[X, y], adv)
            else:
                srch = clone(self.detector_class)
                srch.set_params(cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42))
                srch.fit(np.c_[X, y], adv)
                with open(f"{self.outf}/bayes_{self.exp_name}_{li}.pkl", "wb") as f:
                    pickle.dump(srch, f, pickle.HIGHEST_PROTOCOL)
                det = srch.estimator.set_params(**srch.best_params_).fit(np.c_[X, y], adv)
                self.logger.info(f"{self.exp_name} L{li}: {srch.best_params_}")
            detectors.append(det)
        return detectors

    def get_layer_scores(self, data_loader):
        scores = []
        for li in range(self.n_layers):
            X, y = data_loader(li)
            det = self.detectors[li]
            classes = list(det.classes_)
            adv_idx = classes.index(0) if 0 in classes else 0   
            scores.append(det.predict_proba(np.c_[X, y])[:, adv_idx])
        return np.vstack(scores).T

def _build_registry():
    from sklearn.svm import OneClassSVM
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
    import xgboost as xgb
    import lightgbm as lgb
    from skopt.space import Integer, Real
    return {
        "ocsvm": dict(kind="oneclass", prefix="OCSVM", n_iter=40, 
                      estimator=OneClassSVM(kernel="rbf"),
                      space={"clf__nu": Real(2**-7, 2**-1, prior="log-uniform", base=2),
                             "clf__gamma": Real(2**-15, 2**5, prior="log-uniform", base=2)}),
        "knn": dict(kind="supervised", prefix="KNN", n_iter=20,    
                    estimator=KNeighborsClassifier(),
                    space={"clf__n_neighbors": Integer(1, 15),
                           "clf__weights": ["uniform", "distance"]}),
        "randomforest": dict(kind="supervised", prefix="RF", n_iter=15,   
                    estimator=RandomForestClassifier(n_estimators=100, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200),
                           "clf__max_depth": Integer(3, 20)}),
        "adaboost": dict(kind="supervised", prefix="AB", n_iter=5,       
                    estimator=AdaBoostClassifier(n_estimators=100, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200)}),
        "xgboost": dict(kind="supervised", prefix="XGB", n_iter=40,       
                    estimator=xgb.XGBClassifier(n_estimators=100, max_depth=5, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200),
                           "clf__max_depth": Integer(3, 10)}),
        "lightgbm": dict(kind="supervised", prefix="LGBM", n_iter=40,   
                    estimator=lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200),
                           "clf__max_depth": Integer(3, 10)}),
    }


DEFAULT_N_ITER = {
    "knn": 20, "adaboost": 5, "randomforest": 15,
    "xgboost": 40, "lightgbm": 40, "ocsvm": 40,
}


def run_detector(detector, ds_name, adv_type, net_type="resnet",
                 attacked_root="/kaggle/input/datasets/sealeopard/attacked-pth-files",
                 weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth",
                 out_root="/kaggle/working", n_iter=None, batch_size=100,
                 pre_computed=False, pre_computed_path=None, seed=0, verbose=True):
    from skopt import BayesSearchCV

    spec = _build_registry()[detector]
    if n_iter is None:
        n_iter = DEFAULT_N_ITER.get(detector, 25)
    outf_attacked = f"{attacked_root}/{ds_name.upper()}/{adv_type}"
    out_dir = f"{out_root}/{ds_name}/{detector}/{adv_type}"
    os.makedirs(out_dir, exist_ok=True)
    exp = f"{net_type}_{ds_name}_{adv_type}"

    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.set_device(0)
        torch.cuda.manual_seed_all(seed)

    model, tf = get_model_transforms(net_type, ds_name, n_classes(ds_name), weights_dir)
    ds = DatasetsGA(ds_name, tf, net_type, adv_type, outf_attacked, batch_size)

    clf = Pipeline([("scaler", GroupedScaler()),
                    ("PCA", PCA(whiten=True, random_state=seed)),
                    ("clf", spec["estimator"])])

    if spec["kind"] == "oneclass":
        if pre_computed:
            layer_det = clf
        else:
            split = PredefinedSplit([-1] * len(ds.train_loader.dataset)
                                    + [1] * len(ds.idxs_val))
            layer_det = BayesSearchCV(clf, spec["space"], n_iter=n_iter, n_points=1,
                                      n_jobs=-1, scoring="accuracy", cv=split,
                                      return_train_score=False, refit=False,
                                      random_state=seed, verbose=0)
        trainer = NetDetector(n_layers(net_type), layer_det, exp_name=exp,
                              outf=out_dir, pre_computed=pre_computed,
                              pre_computed_path=pre_computed_path)
    else:
        if pre_computed:
            layer_det = clf
        else:
            layer_det = BayesSearchCV(clf, spec["space"], n_iter=n_iter, n_points=1,
                                      n_jobs=-1, scoring="accuracy",
                                      return_train_score=False, refit=False,
                                      random_state=seed, verbose=0)
        trainer = SupervisedDetector(n_layers(net_type), layer_det, exp_name=exp,
                                     outf=out_dir, pre_computed=pre_computed,
                                     pre_computed_path=pre_computed_path)

    trainer.fit(TrainValLoader(model, ds, batch_size),
                LabelledTrainLoader(model, ds, batch_size),
                ds.adv_test[ds.idxs_train])

    test_scores, output = trainer.predict(LabelledTestLoader(model, ds, batch_size))
    all_output = np.hstack((test_scores, output,
                            ds.adv_test[ds.idxs_test][:, None]))

    prefix = spec["prefix"]
    with open(f"{out_dir}/{prefix}_net_detector_{exp}.pkl", "wb") as f:
        pickle.dump(trainer, f, pickle.HIGHEST_PROTOCOL)
    np.save(f"{out_dir}/{prefix}_{exp}.npy", all_output)

    acc = accuracy_score(ds.adv_test[ds.idxs_test], output[:, 0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:, 1])
    if verbose:
        print(f"[{detector}] {exp}: ACC={acc:.4f}  AUROC={auroc*100:.4f}  -> {out_dir}")
    return out_dir, acc, auroc

import re

def select_best_lid_maha(ds_name, adv_type, net_type="resnet",
                         numpy_root="/kaggle/input/datasets/sealeopard/mahalanobis-and-lid-numpy",
                         out_root="/kaggle/working", verbose=True):
    src = f"{numpy_root}/{net_type}_{ds_name}/{adv_type}"
    out_dir = f"{out_root}/best_{ds_name}"
    os.makedirs(out_dir, exist_ok=True)
    nl = n_layers(net_type)
    results = {}
    for method in ["Mahalanobis", "LID"]:
        regex = r"(\d+)" if method == "LID" else r"(\d+\.\d+|\d+)"
        params = re.findall(f"{method}_{regex}_{ds_name}_{adv_type}",
                            "".join(os.listdir(src)))
        params = sorted(set(params), key=lambda p: float(p))
        best_auc, best_p = -1.0, None
        for p in params:
            data = np.load(f"{src}/{method}_{p}_{ds_name}_{adv_type}.npy")
            adv = np.select([data[:, -1] == 0, data[:, -1] == 1], [0, 1])
            idxs_tr, idxs_val, _, _ = DatasetsGA._split(len(data) // 3)
            lr = LogisticRegressionCV(penalty="l1", solver="liblinear",
                                      max_iter=10000, n_jobs=-1)
            lr.fit(data[idxs_tr][:, :nl], adv[idxs_tr])
            conf = lr.predict_proba(data[idxs_val][:, :nl])[:, 0]
            a = roc_auc_score(adv[idxs_val], -conf)
            if a > best_auc:
                best_auc, best_p = a, p
        X = np.load(f"{src}/{method}_{best_p}_{ds_name}_{adv_type}.npy")
        np.save(f"{out_dir}/{method}_best_{ds_name}_{adv_type}.npy", X)
        results[method] = (best_p, best_auc)
        if verbose:
            print(f"  {method:12s} best param={best_p}  val AUROC={best_auc*100:.4f}%")
    return out_dir, results

In [2]:
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score, precision_recall_curve

ENAD_DETECTORS = [
    ("lid",          "npy"),   
    ("mahalanobis",  "npy"),   
    ("ocsvm",        "pkl"),   
    ("knn",          "pkl"),   
    ("randomforest", "pkl"),   
    ("adaboost",     "pkl"),   
    ("xgboost",      "pkl"),   
    ("lightgbm",     "pkl"),   
]
PKL_PREFIX = {"ocsvm": "OCSVM", "knn": "KNN", "randomforest": "RF",
              "adaboost": "AB", "xgboost": "XGB", "lightgbm": "LGBM"}
ALL_ON = [1] * 8

DEFAULT_ROOTS = {
    "best":       "/kaggle/input/datasets/sealeopard/best-numpy",
    "ocsvm":      "/kaggle/input/datasets/sealeopard/enad-ocsvm-pkl",
    "supervised": "/kaggle/input/datasets/sealeopard/enad-pkl",
}

def _best_npy(roots, method_cap, ds, adv):
    return f"{roots['best']}/best_{ds}/{method_cap}_best_{ds}_{adv}.npy"

def _pkl(roots, detector, ds, adv, net="resnet"):
    prefix = PKL_PREFIX[detector]
    base = (f"{roots['ocsvm']}/{ds}/ocsvm/{adv}" if detector == "ocsvm"
            else f"{roots['supervised']}/{ds}/{detector}/{adv}")
    return f"{base}/{prefix}_net_detector_{net}_{ds}_{adv}.pkl"

def _pkl_npy(roots, detector, ds, adv, net="resnet"):
    prefix = PKL_PREFIX[detector]
    base = (f"{roots['ocsvm']}/{ds}/ocsvm/{adv}" if detector == "ocsvm"
            else f"{roots['supervised']}/{ds}/{detector}/{adv}")
    return f"{base}/{prefix}_{net}_{ds}_{adv}.npy"

def _f1_at_best_threshold(y_true, conf, pos_label=0):
    p, r, _ = precision_recall_curve(y_true, conf, pos_label=pos_label)
    f1 = 2 * p * r / (p + r + 1e-8)
    return float(np.max(f1))

def _metrics(y, conf):
    auroc = roc_auc_score(y, -conf)
    aupr_v = aupr(y, conf, pos_label=0)              
    f1 = _f1_at_best_threshold(y, conf, pos_label=0)
    return float(auroc), float(aupr_v), f1

_MODEL_CACHE = {} 
_DATASET_CACHE = {}   

def clear_caches():
    _MODEL_CACHE.clear(); _DATASET_CACHE.clear()

def build_cache(ds, adv, net="resnet",
                weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth",
                attacked_root="/kaggle/input/datasets/sealeopard/attacked-pth-files",
                roots=DEFAULT_ROOTS, batch_size=100,
                splits=("train", "ga_val", "test"), verbose=True):
    import time
    t0 = time.time()
    if (net, ds) not in _MODEL_CACHE:
        _MODEL_CACHE[(net, ds)] = get_model_transforms(net, ds, n_classes(ds), weights_dir)
    model, tf = _MODEL_CACHE[(net, ds)]
    if (net, ds, adv) not in _DATASET_CACHE:
        _DATASET_CACHE[(net, ds, adv)] = DatasetsGA(
            ds, tf, net, adv, f"{attacked_root}/{ds.upper()}/{adv}", batch_size)
    dataset = _DATASET_CACHE[(net, ds, adv)]
    if verbose:
        print(f"  model+dataset ready ({time.time()-t0:.0f}s)")
    nl = n_layers(net)
    idxs = {"train": dataset.idxs_train, "val": dataset.idxs_val,
            "ga_val": dataset.idxs_ga_val, "test": dataset.idxs_test}
    loaders = {"train": LabelledTrainLoader(model, dataset, batch_size),
               "val": LabelledValLoader(model, dataset, batch_size),
               "ga_val": LabelledGAValLoader(model, dataset, batch_size),
               "test": LabelledTestLoader(model, dataset, batch_size)}

    cache = {i: {} for i in range(8)}
    for i, (name, kind) in enumerate(ENAD_DETECTORS):
        if kind == "npy":
            method = "LID" if name == "lid" else "Mahalanobis"
            arr = np.load(_best_npy(roots, method, ds, adv))
            for s in splits:
                cache[i][s] = arr[idxs[s]][:, :nl]
        else:
            need_pkl = any(s != "test" for s in splits)   
            det = None
            if need_pkl:
                with open(_pkl(roots, name, ds, adv, net), "rb") as f:
                    det = pickle.load(f)
            for s in splits:
                if s == "test":
                    cache[i][s] = np.load(_pkl_npy(roots, name, ds, adv, net))[:, :nl]
                else:
                    cache[i][s] = det.get_layer_scores(loaders[s])[:, :nl]
        if verbose:
            print(f"  cached {name:12s} ({kind})")
    labels = {s: dataset.adv_test[idxs[s]] for s in splits}
    return cache, labels

def _stack(cache, bits, split):
    cols = [cache[i][split] for i in range(8) if bits[i] == 1]
    if not cols:
        raise ValueError("empty subset (all-zero mask)")
    return np.hstack(cols)

def eval_subset(cache, labels, bits, train_split="train", eval_split="test", seed=0):
    Xtr, Xev = _stack(cache, bits, train_split), _stack(cache, bits, eval_split)
    lr = LogisticRegressionCV(penalty="l1", solver="liblinear",
                              max_iter=10000, n_jobs=-1, random_state=seed)
    lr.fit(Xtr, labels[train_split])
    conf = lr.predict_proba(Xev)[:, 0]
    return _metrics(labels[eval_split], conf)

class Solution:
    __slots__ = ("bits", "auroc", "aupr", "f1", "metric")
    def __init__(self, bits):
        self.bits = [int(b) for b in bits]
        self.auroc = self.aupr = self.f1 = self.metric = 0.0

def _fix_zero(bits, rng):
    if sum(bits) == 0:
        bits[int(rng.integers(0, 8))] = 1
    return bits

def genetic_algorithm(fitness_fn, pop_size=8, tournament_size=2, generations=5,
                      seed=0, n_bits=8, parsimony_lambda=0.0, verbose=True):
    rng = np.random.default_rng(seed)

    def evaluate(pop):
        for sol in pop:
            try:
                a, ap, f1 = fitness_fn(sol.bits)
            except Exception as e:
                a = ap = f1 = 0.0
                if verbose:
                    print(f"    fitness error {sol.bits}: {e}")
            sol.auroc, sol.aupr, sol.f1 = a, ap, f1
            sol.metric = a * ap - parsimony_lambda * (sum(sol.bits) / n_bits)

    def tournament(pop):
        idx = rng.choice(len(pop), size=tournament_size, replace=False)
        best = pop[idx[0]]
        for j in idx[1:]:
            if pop[j].metric > best.metric:
                best = pop[j]
        return best

    def crossover(p1, p2):
        o1, o2 = [], []
        for a, b in zip(p1.bits, p2.bits):
            if rng.random() < 0.5:
                o1.append(a); o2.append(b)
            else:
                o1.append(b); o2.append(a)
        return Solution(_fix_zero(o1, rng)), Solution(_fix_zero(o2, rng))

    def mutate(sol, rate=0.4):
        b = [1 - x if rng.random() < rate else x for x in sol.bits]
        return Solution(_fix_zero(b, rng))

    pop = [Solution(_fix_zero(list(rng.integers(0, 2, n_bits)), rng)) for _ in range(pop_size)]
    evaluate(pop)
    for g in range(generations):
        parents = []
        while len(parents) < pop_size // 2:
            a, b = tournament(pop), tournament(pop)
            while a is b:
                b = tournament(pop)
            parents.append((a, b))
        offspring = []
        for a, b in parents:
            c1, c2 = crossover(a, b)
            offspring.extend([mutate(c1), mutate(c2)])
        evaluate(offspring)
        pop = sorted(pop + offspring, key=lambda s: s.metric, reverse=True)[:pop_size]
        if verbose:
            best = pop[0]
            print(f"  gen {g+1}/{generations}: best {best.bits} "
                  f"AUROC={best.auroc*100:.4f} AUPR={best.aupr*100:.4f}")
    return pop[0]

def _fmt(name, m, prec=4):
    return f"  {name:16s} AUROC={m[0]*100:8.{prec}f}  AUPR={m[1]*100:8.{prec}f}  F1={m[2]*100:8.{prec}f}"

def run_normal(ds, adv, net="resnet", weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth",
               attacked_root="/kaggle/input/datasets/sealeopard/attacked-pth-files", roots=DEFAULT_ROOTS,
               mode="ga", custom_bits=None, pop_size=8, generations=5, seed=0,
               parsimony_lambda=0.0, batch_size=100):
    print(f"\n===== NORMAL [{mode}]: {ds} / {adv} =====")
    cache, labels = build_cache(ds, adv, net, weights_dir, attacked_root, roots,
                                batch_size, splits=("train", "ga_val", "test"))
    if mode == "full":
        m = eval_subset(cache, labels, ALL_ON, "train", "test", seed)
        print(_fmt("full stacking", m)); return {"mode": "full", "bits": ALL_ON, "metrics": m}
    if mode == "custom":
        if custom_bits is None:
            raise ValueError("mode='custom' requires custom_bits")
        m = eval_subset(cache, labels, custom_bits, "train", "test", seed)
        print(_fmt(f"custom {custom_bits}", m)); return {"mode": "custom", "bits": list(custom_bits), "metrics": m}
    if mode == "ga":
        best = genetic_algorithm(
            lambda bits: eval_subset(cache, labels, bits, "train", "ga_val", seed),
            pop_size=pop_size, generations=generations, seed=seed, parsimony_lambda=parsimony_lambda)
        m = eval_subset(cache, labels, best.bits, "train", "test", seed)
        print(_fmt(f"GA {best.bits}", m)); return {"mode": "ga", "bits": best.bits, "metrics": m}
    raise ValueError(f"unknown mode: {mode!r} (use 'ga' | 'full' | 'custom')")


def _lr_transfer(seed=0):
    return LogisticRegressionCV(penalty="l2", max_iter=5000, n_jobs=-1, random_state=seed)


def eval_transfer_select(src_cache, src_labels, bits, seed=0):
    Xtr = np.vstack([_stack(src_cache, bits, "train"), _stack(src_cache, bits, "val")])
    ytr = np.concatenate([src_labels["train"], src_labels["val"]])
    Xev = _stack(src_cache, bits, "ga_val")
    lr = _lr_transfer(seed); lr.fit(Xtr, ytr)
    conf = lr.predict_proba(Xev)[:, 0]
    return _metrics(src_labels["ga_val"], conf)


def eval_transfer_test(src_cache, src_labels, tgt_cache, tgt_labels, bits, seed=0):
    Xtr = np.vstack([_stack(src_cache, bits, "train"), _stack(src_cache, bits, "val")])
    ytr = np.concatenate([src_labels["train"], src_labels["val"]])
    Xev = _stack(tgt_cache, bits, "test")
    lr = _lr_transfer(seed); lr.fit(Xtr, ytr)
    conf = lr.predict_proba(Xev)[:, 0]
    return _metrics(tgt_labels["test"], conf)


def run_transfer(ds, adv_source, adv_target, net="resnet",
                 weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth",
                 attacked_root="/kaggle/input/datasets/sealeopard/attacked-pth-files", roots=DEFAULT_ROOTS,
                 mode="ga", custom_bits=None, pop_size=8, generations=5, seed=0,
                 parsimony_lambda=0.0, batch_size=100):
    print(f"\n===== TRANSFER [{mode}]: train {adv_source} -> test {adv_target}  ({ds}) =====")
    src_cache, src_labels = build_cache(ds, adv_source, net, weights_dir, attacked_root, roots,
                                        batch_size, splits=("train", "val", "ga_val"),
                                        verbose=False)
    tgt_cache, tgt_labels = build_cache(ds, adv_target, net, weights_dir, attacked_root, roots,
                                        batch_size, splits=("test",), verbose=False)

    def final(bits):
        return eval_transfer_test(src_cache, src_labels, tgt_cache, tgt_labels, bits, seed)

    if mode == "full":
        m = final(ALL_ON)
        print(_fmt("full stacking", m))
        return {"mode": "full", "bits": ALL_ON, "metrics": m}

    if mode == "custom":
        if custom_bits is None:
            raise ValueError("mode='custom' requires custom_bits")
        m = final(custom_bits)
        print(_fmt(f"custom {custom_bits}", m))
        return {"mode": "custom", "bits": list(custom_bits), "metrics": m}

    if mode == "ga":
        best = genetic_algorithm(
            lambda bits: eval_transfer_select(src_cache, src_labels, bits, seed),
            pop_size=pop_size, generations=generations, seed=seed,
            parsimony_lambda=parsimony_lambda)
        m = final(best.bits)
        print(_fmt(f"GA {best.bits}", m))
        return {"mode": "ga", "bits": best.bits, "metrics": m}

    raise ValueError(f"unknown mode: {mode!r} (use 'ga' | 'full' | 'custom')")

In [3]:
import os
import json
import pickle
import numpy as np
import torch
import sklearn.covariance
from scipy.spatial.distance import cdist
from sklearn.linear_model import LogisticRegressionCV

_STD_T = torch.tensor(CIFAR_STD, device=DEVICE).view(1, 3, 1, 1)  

ENAD_BASELINE = [1, 1, 1, 0, 0, 0, 0, 0]
GA_COMBOS = {
    "cifar10": {"FGSM": [0, 0, 0, 0, 0, 1, 0, 0], "BIM": [0, 1, 0, 0, 0, 1, 1, 0],
                "DeepFool": [0, 1, 1, 1, 1, 1, 1, 0], "CWL2": [0, 1, 1, 1, 0, 1, 1, 1]},
    "svhn":    {"FGSM": [1, 0, 0, 0, 1, 0, 0, 0], "BIM": [0, 0, 0, 0, 0, 1, 0, 0],
                "DeepFool": [1, 1, 1, 1, 0, 1, 1, 0], "CWL2": [1, 1, 0, 1, 1, 1, 1, 1]},
}

def combo_for(mode, ds, adv):
    if mode == "enad":
        return list(ENAD_BASELINE)         
    if mode == "enad_full":
        return [1] * 8                     
    return list(GA_COMBOS[ds][adv])       

def sample_estimator(model, num_classes, dims, train_loader):
    nl = len(dims)
    feats = [[] for _ in range(nl)]
    labels = []
    with torch.no_grad():
        for data, target in train_loader:
            data = data.to(DEVICE)
            _, outs = model.feature_list(data)
            for k in range(nl):
                f = outs[k].view(outs[k].size(0), outs[k].size(1), -1).mean(2)
                feats[k].append(f.cpu())
            labels.append(target.cpu())
    labels = torch.cat(labels)
    feats = [torch.cat(fl, 0) for fl in feats]
    sample_mean, precision = [], []
    lasso = sklearn.covariance.EmpiricalCovariance(assume_centered=False)
    for k in range(nl):
        Fk = feats[k]
        means = torch.stack([Fk[labels == c].mean(0) for c in range(num_classes)])
        centered = Fk - means[labels]
        lasso.fit(centered.numpy())
        sample_mean.append(means.to(DEVICE))
        precision.append(torch.from_numpy(lasso.precision_).float().to(DEVICE))
    return sample_mean, precision

def _gaussian_scores(feat, mean_c, prec):
    cols = []
    for c in range(mean_c.size(0)):
        zf = feat - mean_c[c]
        cols.append(-0.5 * ((zf @ prec) * zf).sum(1))
    return torch.stack(cols, 1)

def mle_batch(data, batch, k):
    data = np.asarray(data, dtype=np.float32)
    batch = np.asarray(batch, dtype=np.float32)
    k = min(k, len(data) - 1)
    f = lambda v: -k / np.sum(np.log(v / v[-1]))
    a = cdist(batch, data)
    a = np.sort(a, axis=1)[:, 1:k + 1]
    return np.apply_along_axis(f, axis=1, arr=a)

def maha_image_scores(model, img, sample_mean, precision, magnitude, nl):
    model.eval()
    scores = []
    for li in range(nl):
        data = img.clone().detach().requires_grad_(True)
        feat = model.intermediate_forward(data, li)
        feat = feat.view(feat.size(0), feat.size(1), -1).mean(2)
        g = _gaussian_scores(feat, sample_mean[li], precision[li])
        pred = g.argmax(1)
        zf = feat - sample_mean[li][pred]
        pure = -0.5 * ((zf @ precision[li]) * zf).sum(1)
        loss = (-pure).mean()
        loss.backward()
        grad = (data.grad.detach() >= 0).float() * 2 - 1
        grad = grad / _STD_T
        temp = data.detach() - magnitude * grad
        with torch.no_grad():
            nf = model.intermediate_forward(temp, li)
            nf = nf.view(nf.size(0), nf.size(1), -1).mean(2)
            ng = _gaussian_scores(nf, sample_mean[li], precision[li])
            scores.append(float(ng.max(1).values.cpu()))
    return np.array(scores, dtype=np.float32)

def lid_image_scores(model, img, lid_ref, k, nl):
    with torch.no_grad():
        _, outs = model.feature_list(img)
    scores = []
    for li in range(nl):
        f = outs[li].view(outs[li].size(0), outs[li].size(1), -1).mean(2).cpu().numpy()  
        scores.append(float(mle_batch(lid_ref[li], f, k)[0]))
    return np.array(scores, dtype=np.float32)

class SingleImageLoader:
    def __init__(self, model, img):
        self.model, self.img = model, img         
    def __call__(self, layer_idx):
        acts, pred = extract_activations(self.img, self.model, layer_idx, return_pred=True)
        return acts.cpu().numpy(), pred.cpu().numpy()

def _feature_dims(model):
    dummy = torch.rand(2, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        _, outs = model.feature_list(dummy)
    return [o.size(1) for o in outs]

def build_demo_assets(datasets, attacks, net="resnet",
                      weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth",
                      attacked_root="/kaggle/input/datasets/sealeopard/attacked-pth-files",
                      roots=DEFAULT_ROOTS,
                      numpy_root="/kaggle/input/datasets/sealeopard/mahalanobis-and-lid-numpy",
                      out_root="/kaggle/working/demo_assets",
                      lid_ref_size=5000, seed=0, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    os.makedirs(out_root, exist_ok=True)
    bestmk = {}

    for ds in datasets:
        model, tf = get_model_transforms(net, ds, n_classes(ds), weights_dir)
        dims = _feature_dims(model)
        train_loader, _ = data_loader.getTargetDataSet(ds, 100, tf, DATA_ROOT)

        sm, prec = sample_estimator(model, n_classes(ds), dims, train_loader)
        with open(f"{out_root}/maha_stats_{ds}.pkl", "wb") as f:
            pickle.dump(([m.cpu() for m in sm], [p.cpu() for p in prec]), f)

        feats = [[] for _ in dims]
        seen = 0
        with torch.no_grad():
            for data, _ in train_loader:
                _, outs = model.feature_list(data.to(DEVICE))
                for li in range(len(dims)):
                    feats[li].append(outs[li].view(outs[li].size(0), outs[li].size(1), -1)
                                     .mean(2).cpu().numpy())
                seen += data.size(0)
                if seen >= lid_ref_size:
                    break
        lid_ref = [np.concatenate(fl)[:lid_ref_size] for fl in feats]
        with open(f"{out_root}/lid_ref_{ds}.pkl", "wb") as f:
            pickle.dump(lid_ref, f)
        if verbose:
            print(f"[{ds}] saved maha_stats + lid_ref ({lid_ref[0].shape[0]} refs)")

        for adv in attacks:
            _, results = select_best_lid_maha(ds, adv, net_type=net,
                                              numpy_root=numpy_root, out_root="/kaggle/working",
                                              verbose=False)
            best_m = float(results["Mahalanobis"][0])
            best_k = int(results["LID"][0])
            bestmk[f"{ds}_{adv}"] = {"m": best_m, "k": best_k}

            cache, labels = build_cache(ds, adv, net, weights_dir, attacked_root, roots,
                                        splits=("train",), verbose=False)
            for mode in ["enad", "enad_full", "enad_ga"]:
                bits = combo_for(mode, ds, adv)
                X = _stack(cache, bits, "train")
                lr = LogisticRegressionCV(penalty="l1", solver="liblinear",
                                          max_iter=10000, n_jobs=-1, random_state=seed)
                lr.fit(X, labels["train"])
                with open(f"{out_root}/lr_{ds}_{adv}_{mode}.pkl", "wb") as f:
                    pickle.dump({"lr": lr, "bits": bits}, f)
            if verbose:
                print(f"  [{ds}/{adv}] best m={best_m} k={best_k}; saved enad + enad_full + enad_ga heads")

    with open(f"{out_root}/best_mk.json", "w") as f:
        json.dump(bestmk, f, indent=2)

    import shutil
    for ds in datasets:
        shutil.rmtree(f"/kaggle/working/best_{ds}", ignore_errors=True)

    if verbose:
        print("saved best_mk.json ->", out_root)
    return out_root

def load_demo_assets(ds, net, weights_dir, assets_dir):
    model, tf = get_model_transforms(net, ds, n_classes(ds), weights_dir)
    with open(f"{assets_dir}/maha_stats_{ds}.pkl", "rb") as f:
        sm_cpu, prec_cpu = pickle.load(f)
    sm = [m.to(DEVICE) for m in sm_cpu]; prec = [p.to(DEVICE) for p in prec_cpu]
    with open(f"{assets_dir}/lid_ref_{ds}.pkl", "rb") as f:
        lid_ref = pickle.load(f)
    with open(f"{assets_dir}/best_mk.json") as f:
        bestmk = json.load(f)
    return {"model": model, "tf": tf, "sm": sm, "prec": prec,
            "lid_ref": lid_ref, "bestmk": bestmk}

def predict_image(pil_img, ds, adv, mode, assets, net="resnet", roots=DEFAULT_ROOTS,
                  assets_dir="/kaggle/input/datasets/sealeopard/enad-demo-assets"):
    model, tf = assets["model"], assets["tf"]
    nl = n_layers(net)
    img = tf(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)

    bits = combo_for(mode, ds, adv)
    mk = assets["bestmk"][f"{ds}_{adv}"]
    feats, used = [], []

    if bits[0]:
        feats.append(lid_image_scores(model, img, assets["lid_ref"], mk["k"], nl)); used.append("LID")
    if bits[1]:
        feats.append(maha_image_scores(model, img, assets["sm"], assets["prec"], mk["m"], nl)); used.append("Maha")
    pkl_names = {2: "ocsvm", 3: "knn", 4: "randomforest", 5: "adaboost", 6: "xgboost", 7: "lightgbm"}
    loader = SingleImageLoader(model, img)
    for idx, name in pkl_names.items():
        if bits[idx]:
            with open(_pkl(roots, name, ds, adv, net), "rb") as f:
                det = pickle.load(f)
            feats.append(det.get_layer_scores(loader)[0, :nl]); used.append(name)

    X = np.hstack(feats).reshape(1, -1)
    with open(f"{assets_dir}/lr_{ds}_{adv}_{mode}.pkl", "rb") as f:
        head = pickle.load(f)
    lr = head["lr"]
    p_adv = float(lr.predict_proba(X)[0, 0])    
    label = "adversarial" if lr.predict(X)[0] == 0 else "normal"
    return label, p_adv, used

In [4]:
import warnings; warnings.filterwarnings("ignore")
!pip install -q gradio
setup()
print("Device:", DEVICE)

NET_TYPE    = "resnet"
WEIGHTS_DIR = "/kaggle/input/datasets/sealeopard/resnet-pth"                                   # <-- your weights path
ASSETS_DIR  = "/kaggle/input/datasets/sealeopard/enad-demo-assets"         
ROOTS = {
    "best":       "/kaggle/input/datasets/sealeopard/best-numpy",
    "ocsvm":      "/kaggle/input/datasets/sealeopard/enad-ocsvm-pkl",
    "supervised": "/kaggle/input/datasets/sealeopard/enad-pkl",
}

_ASSETS = {ds: load_demo_assets(ds, NET_TYPE, WEIGHTS_DIR, ASSETS_DIR)
           for ds in ["cifar10", "svhn"]}
print("assets loaded for:", list(_ASSETS))

Device: cpu
assets loaded for: ['cifar10', 'svhn']


In [5]:
import gradio as gr

def run(image, ds_name, adv_type, algorithm):
    if image is None:
        return "Please upload an image.", None
    mode = {"ENAD (LID+Maha+OCSVM)": "enad",
            "ENAD-full": "enad_full",
            "ENAD-GA (selected subset)": "enad_ga"}[algorithm]
    try:
        label, p_adv, used = predict_image(
            image, ds_name, adv_type, mode, _ASSETS[ds_name],
            net=NET_TYPE, roots=ROOTS, assets_dir=ASSETS_DIR)
    except Exception as e:
        return f"Error: {e}", None
    verdict = f"Prediction: {label.upper()}   (P[adversarial] = {p_adv:.3f})"
    detail = {"adversarial": p_adv, "normal": 1.0 - p_adv}
    print(f"{ds_name}/{adv_type}/{mode} detectors={used} -> {label} ({p_adv:.3f})")
    return verdict, detail

with gr.Blocks(title="ENAD Adversarial Detector") as demo:
    gr.Markdown("## ENAD — Adversarial Image Detector\n"
                "Upload an image, pick the dataset / attack / method, and Predict.")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="pil", label="Image (32x32)")
            ds = gr.Dropdown(["cifar10", "svhn"], value="cifar10", label="Dataset")
            adv = gr.Dropdown(["FGSM", "BIM", "DeepFool", "CWL2"], value="FGSM", label="Attack the detector was tuned for")
            alg = gr.Radio(["ENAD (LID+Maha+OCSVM)", "ENAD-full", "ENAD-GA (selected subset)"],
                           value="ENAD-GA (selected subset)", label="Method")
            btn = gr.Button("Predict", variant="primary")
        with gr.Column():
            out_txt = gr.Textbox(label="Result")
            out_lbl = gr.Label(label="Confidence")
    btn.click(run, inputs=[inp, ds, adv, alg], outputs=[out_txt, out_lbl])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0198186d661766a0d8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
